# Phase 8 — RAG Triage from Top-K Fused Results (v3.0)

This notebook implements a **RAG (Retrieval-Augmented Generation)** pipeline
that reads the top-K fused results produced by **`scripts/08_rag_top_k_extraction.py`**
and generates structured intelligence briefs using a local LLM via Ollama.

**Input:** `08_(RAG)_top_{K}_fused_with_full_data_{timestamp}.xlsx`

**Pipeline:**
1. Load top-K fused results (enriched with full document data)
2. Format each record for LLM consumption
3. Generate structured triage notes per query via Ollama
4. Run structural audit on each output
5. Export results


---
## 0 · Configuration

In [1]:
# ======================================================================
# CONFIGURATION
# ======================================================================

# Path to the RAG input file from 08_rag_top_k_extraction.py
# Uses the most recent 08_(RAG)_*.xlsx in results/ if not set explicitly.
RAG_INPUT_FILE = None  # Set to a specific path to override auto-detect

# ======================================================================
# LLM — gpt-oss:20b via Ollama (local, free, offline)
# ======================================================================
LLM_MODEL = 'gpt-oss:20b'
LLM_TEMPERATURE = 0.0
OLLAMA_BASE_URL = 'http://localhost:11434'
OLLAMA_TIMEOUT = 600  # 10 min timeout — 20B model can be slow

# Max results per query to send to the LLM
MAX_RECORDS_PER_QUERY = 10

# RAGAS evaluation — requires paid API, set to None to skip
EVAL_LLM_MODEL = None

print(f'RAG input file : {RAG_INPUT_FILE or "(auto-detect most recent)"}')
print(f'LLM model      : {LLM_MODEL}')
print(f'Ollama URL     : {OLLAMA_BASE_URL}')
print(f'Temperature    : {LLM_TEMPERATURE}')
print(f'Timeout        : {OLLAMA_TIMEOUT}s')
print(f'Max records/q  : {MAX_RECORDS_PER_QUERY}')


RAG input file : (auto-detect most recent)
LLM model      : gpt-oss:20b
Ollama URL     : http://localhost:11434
Temperature    : 0.0
Timeout        : 600s
Max records/q  : 10


---
## 1 · Imports

In [2]:
import os
import re
import json
import time
import warnings
from pathlib import Path
from collections import Counter
from datetime import datetime

import pandas as pd
import requests

warnings.filterwarnings('ignore')

print("Imports loaded.")

Imports loaded.


---
## 2 · Connect to gpt-oss:20b via Ollama

Connects to the Ollama server running locally. Make sure:
1. Ollama is running
2. You have pulled the model: `ollama pull gpt-oss:20b`
3. Verify with: `ollama list`

In [3]:
# ======================================================================
# LLM Client — gpt-oss:20b via Ollama
# ======================================================================

def llm_generate(system_prompt, user_prompt,
                 temperature=LLM_TEMPERATURE, model=LLM_MODEL):
    """
    Generate text using gpt-oss:20b via Ollama's chat API.
    """
    try:
        resp = requests.post(
            f"{OLLAMA_BASE_URL}/api/chat",
            json={
                "model": model,
                "stream": False,
                "options": {"temperature": temperature},
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
            },
            timeout=OLLAMA_TIMEOUT,
        )
        resp.raise_for_status()
        return resp.json()["message"]["content"]

    except requests.ConnectionError:
        raise ConnectionError(
            f"Cannot connect to Ollama at {OLLAMA_BASE_URL}.\n"
            f"  1. Install Ollama: https://ollama.com\n"
            f"  2. Pull model: ollama pull {model}\n"
            f"  3. Verify: ollama list\n"
            f"  4. Make sure Ollama is running"
        )
    except requests.exceptions.HTTPError as e:
        raise RuntimeError(
            f"Ollama returned an error: {e}\n"
            f"  This usually means the model is not pulled.\n"
            f"  Run: ollama pull {model}\n"
            f"  Then verify with: ollama list"
        )
    except requests.exceptions.Timeout:
        raise TimeoutError(
            f"Ollama timed out after {OLLAMA_TIMEOUT}s.\n"
            f"  The 20B model can be slow on CPU.\n"
            f"  Increase OLLAMA_TIMEOUT in the config cell if needed."
        )


# ---- Connectivity test ----
print(f"Connecting to Ollama at {OLLAMA_BASE_URL}...")
print(f"Model: {LLM_MODEL}")
_test = llm_generate("You are a test.", "Say OK and nothing else.")
print(f"Response: '{_test.strip()[:50]}'")
print(f"gpt-oss:20b connected and ready.")

Connecting to Ollama at http://localhost:11434...
Model: gpt-oss:20b


Response: 'OK'
gpt-oss:20b connected and ready.


---
## 3 · Load RAG Input File

Reads the enriched top-K fused results produced by `scripts/08_rag_top_k_extraction.py`.
Each row is a `(query_id, doc_id)` pair with fused rank, query text, and full document data.


In [4]:
# ======================================================================
# Load the RAG input file from 08_rag_top_k_extraction.py
# ======================================================================

from pathlib import Path

rag_input_path = None

if RAG_INPUT_FILE is not None:
    rag_input_path = Path(RAG_INPUT_FILE)
else:
    # Auto-detect: find the most recent 08_(RAG)_*.xlsx in results/
    search_dirs = [
        Path('results'),
        Path('../results'),
        Path.cwd() / 'results',
    ]
    for search_dir in search_dirs:
        if search_dir.exists():
            candidates = sorted(search_dir.glob('08_(RAG)_*.xlsx'))
            if candidates:
                rag_input_path = candidates[-1]  # most recent
                break

if rag_input_path is None or not rag_input_path.exists():
    raise FileNotFoundError(
        'Could not find 08_(RAG)_*.xlsx in results/.\n'
        '  Run: python scripts/08_rag_top_k_extraction.py\n'
        '  Or set RAG_INPUT_FILE in the config cell.'
    )

# Load the All_Combined sheet (contains all query types)
df = pd.read_excel(rag_input_path, sheet_name='All_Combined')

print(f'RAG input loaded: {rag_input_path}')
print(f'  Rows: {len(df):,}')
print(f'  Queries: {df["query_id"].nunique()}')
print(f'  Columns: {list(df.columns)}')
print()

# Derive query_type from query_id prefix (e.g. Q3_001 -> Type 3)
df['query_type'] = df['query_id'].str.extract(r'Q(\d+)_')[0].apply(lambda x: f'Type {x}')
df['query_notes'] = ''  # No notes column in fused output; set empty

# Show sample
print('Sample row:')
print(df.iloc[0].to_string())


RAG input loaded: ..\results\08_(RAG)_top_10_fused_with_full_data_20260410_091119.xlsx
  Rows: 32
  Queries: 6
  Columns: ['query_id', 'doc_id', 'fused_rank', 'rrf_score', 'query_text', 'caption', 'schema', 'text_blob', 'embedding_text', 'first_seen', 'last_seen', 'token_count', 'id_innCode', 'id_ogrnCode', 'id_kppCode', 'meta_country', 'meta_programId', 'meta_datasets', 'id_imoNumber', 'id_mmsi', 'id_callSign', 'id_registrationNumber', 'id_npiCode']

Sample row:
query_id                                                            Q1_002
doc_id                      sk-po-354d893265060ef749116a7722d8880e06000be5
fused_rank                                                               1
rrf_score                                                         0.032787
query_text                       PW2XZT68KVW9, PW2XZT69KVW8, PW 2XZT68KVW9
caption                                                       Pavol Mertus
schema                                                              Person
text_bl

---
## 4 · Format Records for LLM

Converts each row in the pooling file into a structured text block
that the LLM can read as a "search result".

In [5]:
def format_pooling_row_for_llm(row, rank):
    """
    Convert a single DataFrame row into a structured record
    for the RAG prompt. Adapted for 08_rag_top_k_extraction.py output.
    """
    lines = []

    # Source / datasets
    source = row.get('meta_datasets', '')
    if pd.isna(source) or not source:
        source = 'Unknown Source'
    else:
        source = str(source).split(',')[0].strip()

    # RRF score
    score = row.get('rrf_score', '')
    if pd.isna(score):
        score = 'N/A'

    lines.append(f'[RECORD {rank} — SOURCE: {source} | RRF Score: {score}]')

    # Name
    caption = row.get('caption', 'N/A')
    if pd.notna(caption):
        lines.append(f'Name: {caption}')

    # Entity type
    schema = row.get('schema', '')
    if pd.notna(schema) and schema:
        lines.append(f'Entity Type: {schema}')

    # Country
    country = row.get('meta_country', '')
    if pd.notna(country) and country:
        lines.append(f'Country: {country}')

    # Programs / Sanctions lists
    program = row.get('meta_programId', '')
    if pd.notna(program) and program:
        lines.append(f'Sanctions Programs: {str(program)[:300]}')

    # Datasets
    datasets = row.get('meta_datasets', '')
    if pd.notna(datasets) and datasets:
        lines.append(f'Listed In: {str(datasets)[:300]}')

    # ID fields
    id_fields = {
        'id_callSign': 'Call Sign',
        'id_idNumber': 'ID Number',
        'id_imoNumber': 'IMO Number',
        'id_innCode': 'INN Code',
        'id_kppCode': 'KPP Code',
        'id_mmsi': 'MMSI',
        'id_ogrnCode': 'OGRN Code',
        'id_registrationNumber': 'Registration Number',
        'id_uniqueEntityId': 'Unique Entity ID',
        'id_taxNumber': 'Tax Number',
        'id_email': 'Email',
        'id_phone': 'Phone',
        'id_passportNumber': 'Passport Number',
        'id_leiCode': 'LEI Code',
        'id_npiCode': 'NPI Code',
    }
    for col, label in id_fields.items():
        val = row.get(col, '')
        if pd.notna(val) and str(val).strip():
            lines.append(f'{label}: {str(val)[:200]}')

    # Text content
    text_preview = row.get('embedding_text', row.get('text_blob', ''))
    if pd.notna(text_preview) and str(text_preview).strip():
        preview = str(text_preview)[:800]
        lines.append(f'Text Details: {preview}')

    # Dates
    for date_col, label in [('first_seen', 'First Seen'), ('last_seen', 'Last Seen')]:
        val = row.get(date_col, '')
        if pd.notna(val) and str(val).strip():
            lines.append(f'{label}: {str(val)[:30]}')

    return '\n'.join(lines)


# ---- Preview with first query ----
first_qid = df['query_id'].iloc[0]
sample_rows = df[df['query_id'] == first_qid].head(2)
for idx, (_, row) in enumerate(sample_rows.iterrows(), 1):
    print(f'=== RECORD {idx} PREVIEW ===')
    print(format_pooling_row_for_llm(row, idx))
    print()


=== RECORD 1 PREVIEW ===
[RECORD 1 — SOURCE: sk_public_officials | RRF Score: 0.032787]
Name: Pavol Mertus
Entity Type: Person
Country: sk
Listed In: sk_public_officials
Text Details: Pavol Mertus. Type: Person. Country: Slovakia. Listed in: Sk Public Officials.
First Seen: 2025-01-03T11:03:33
Last Seen: 2026-03-26T05:17:02

=== RECORD 2 PREVIEW ===
[RECORD 2 — SOURCE: sk_public_officials | RRF Score: 0.032258]
Name: Lenka Hmírová
Entity Type: Person
Country: sk
Listed In: sk_public_officials
Text Details: Lenka Hmírová. Type: Person. Country: Slovakia. Listed in: Sk Public Officials.
First Seen: 2025-01-03T11:03:33
Last Seen: 2026-03-26T05:17:02



---
## 5 · The Hardened Sanctions Screening Prompt

The system prompt enforces faithfulness rules, and the user prompt
template structures the output for each query.

In [6]:
# ======================================================================
# SYSTEM PROMPT — sanctions intelligence analyst
# ======================================================================

SYSTEM_PROMPT = """You are a sanctions intelligence analyst. You are reviewing search results
retrieved for an analytical query. Your task is to assess each result
against the query criteria and produce a structured intelligence brief.

RULES:

1. Only use information present in the search results below.
   Do not add facts from your own knowledge, even if you are certain
   they are true.

2. For each record, explicitly state which query criteria it satisfies
   and which it does not, citing the specific fields from the record.

3. When a record lacks information needed to assess a criterion,
   state "NOT DETERMINABLE FROM RECORD" — do not assume.

4. Use [Record N] tags to attribute every factual claim.
"""


# ======================================================================
# USER PROMPT TEMPLATE
# ======================================================================

USER_PROMPT_TEMPLATE = """Analyze the following search results against the query criteria.

QUERY:
Search terms: {query_name}
Query type: {query_type}
Analyst notes: {query_notes}

SEARCH RESULTS:
{formatted_records}

---

REQUIRED OUTPUT:

### CRITERIA EXTRACTION

First, break down the query into its individual criteria:
- Criterion 1: [extracted from query]
- Criterion 2: [extracted from query]
- Criterion 3: [extracted from query]

### PER-RECORD ASSESSMENT

For EACH record:

**Record [N]: [Entity name] | [Entity type] | [Country]**

Criteria met:
- Criterion 1: [MET / NOT MET / NOT DETERMINABLE] — [evidence from record]
- Criterion 2: [MET / NOT MET / NOT DETERMINABLE] — [evidence from record]
- Criterion 3: [MET / NOT MET / NOT DETERMINABLE] — [evidence from record]

Key facts from record:
- [fact 1] [Record N]
- [fact 2] [Record N]

Relevance: [RELEVANT / PARTIALLY RELEVANT / NOT RELEVANT]

### CROSS-RECORD PATTERNS

Identify any connections, patterns, or groupings across the relevant records:
- Common sanctions programs
- Shared jurisdictions or countries
- Related entities mentioned across multiple records
- Common designation reasons

If no patterns are identifiable from the records, state:
"No cross-record patterns identifiable from the provided data."

### SUMMARY

Total records assessed: [N]
Relevant: [N]
Partially relevant: [N]
Not relevant: [N]

Brief (2-3 sentences): Summarize what the relevant records collectively
reveal about the query topic, using only facts from the records.

### GAPS AND LIMITATIONS

What information is missing that would improve this analysis?
What aspects of the query could not be fully addressed by these records?
"""

print(f"System prompt : {len(SYSTEM_PROMPT):,} chars")
print(f"User template : {len(USER_PROMPT_TEMPLATE):,} chars")

System prompt : 697 chars
User template : 1,655 chars


---
## 6 · RAG Pipeline

Groups the pooling rows by query, formats them, sends to LLM, returns results.

In [7]:
def run_rag_for_query(query_group_df):
    """
    Run RAG triage for a single query_id group from the pooling file.

    Parameters
    ----------
    query_group_df : DataFrame — rows from pooling file for one query_id

    Returns
    -------
    dict with query info, formatted records, generated note, timing
    """
    # Extract query info from first row
    first_row = query_group_df.iloc[0]
    query_id = str(first_row.get('query_id', ''))
    query_text = str(first_row.get('query_text', ''))
    query_type = str(first_row.get('query_type', ''))
    query_notes = str(first_row.get('query_notes', ''))

    # Limit records per query
    records_df = query_group_df.head(MAX_RECORDS_PER_QUERY)

    # Format each row as a structured record
    formatted_blocks = []
    for rank, (_, row) in enumerate(records_df.iterrows(), 1):
        block = format_pooling_row_for_llm(row, rank)
        formatted_blocks.append(block)

    formatted_records = "\n\n".join(formatted_blocks)

    # Build prompt
    user_prompt = USER_PROMPT_TEMPLATE.format(
        query_name=query_text,
        query_type=query_type,
        query_notes=query_notes if query_notes != 'nan' else 'None',
        query_dob="Not provided by requesting party",
        query_nationality="Not provided by requesting party",
        query_identifiers="None provided",
        formatted_records=formatted_records,
    )

    # Generate
    t0 = time.time()
    generated_note = llm_generate(SYSTEM_PROMPT, user_prompt)
    latency = time.time() - t0

    return {
        'query_id': query_id,
        'query_text': query_text,
        'query_type': query_type,
        'query_notes': query_notes,
        'n_records': len(records_df),
        'formatted_records': formatted_records,
        'user_prompt': user_prompt,
        'generated_note': generated_note,
        'latency_seconds': latency,
    }


print("RAG pipeline function defined.")

RAG pipeline function defined.


---
## 7 · Structural Audit (Evaluation Layer 1)

Free, instant checks on every output — no LLM needed.

In [8]:
def structural_audit(generated_output, formatted_records):
    """
    Automated structural checks on a generated triage note.
    Returns dict with pass/fail, issues, and scores.
    """
    issues = []

    # 1. Banned phrases
    banned = [
        "is known to", "is believed to", "reportedly",
        "it is well known", "based on available information",
    ]
    found_banned = [p for p in banned if p in generated_output.lower()]
    if found_banned:
        issues.append({"type": "BANNED_PHRASE", "severity": "CRITICAL",
                       "phrases": found_banned})

    # 2. Phantom sources
    cited = set(re.findall(r'\[Record (\d+)\]', generated_output, re.IGNORECASE))
    n_input = len(re.findall(r'\[RECORD \d+', formatted_records))
    phantom = [r for r in cited if int(r) > n_input]
    if phantom:
        issues.append({"type": "PHANTOM_SOURCE", "severity": "CRITICAL",
                       "phantom_records": phantom, "max_valid": n_input})

    # 3. Weak TRUE POSITIVE
    tp_sections = re.findall(
        r'CLASSIFICATION:\s*TRUE POSITIVE(.*?)(?=CLASSIFICATION:|###|$)',
        generated_output, re.DOTALL | re.IGNORECASE
    )
    weak_tp = False
    for section in tp_sections:
        matches = re.findall(r'—\s*match', section.lower())
        if len(matches) < 2:
            weak_tp = True
            issues.append({"type": "WEAK_TRUE_POSITIVE", "severity": "CRITICAL",
                           "detail": "TRUE POSITIVE with < 2 field matches"})

    # 4. Ungrounded "is associated with"
    assoc = re.findall(r'[^.]*is associated with[^.]*\.', generated_output, re.IGNORECASE)
    for match in assoc:
        if not re.search(r'\[Record \d+\]', match, re.IGNORECASE):
            issues.append({"type": "UNGROUNDED_ASSOCIATION", "severity": "WARNING",
                           "sentence": match.strip()[:150]})

    # Scores
    scores = {
        'no_banned_phrases': 1 if not found_banned else 0,
        'no_phantom_sources': 1 if not phantom else 0,
        'no_weak_true_positive': 0 if weak_tp else 1,
    }
    scores['structural_score'] = round(sum(scores.values()) / 3, 4)

    return {
        "pass": len([i for i in issues if i.get('severity') == 'CRITICAL']) == 0,
        "issues": issues,
        "issue_count": len(issues),
        "scores": scores,
    }


print("Structural audit defined.")

Structural audit defined.


---
## 8 · Run RAG on All Queries

Iterates through each unique `query_id` in the pooling file, generates a
triage note, and runs the structural audit.

> **Timing:** gpt-oss:20b at 20B parameters will take ~30-120 seconds per query
> depending on GPU/CPU and number of records. Be patient.

In [9]:
# ======================================================================
# Process all queries
# ======================================================================

query_ids = df['query_id'].unique()
print(f"Total queries to process: {len(query_ids)}")
print(f"{'='*80}\n")

all_results = []
all_evals = []

for i, qid in enumerate(query_ids, 1):
    query_df = df[df['query_id'] == qid]
    query_text = str(query_df.iloc[0].get('query_text', qid))
    print(f"[{i}/{len(query_ids)}] {qid}: {query_text[:60]}...")

    # Run RAG
    result = run_rag_for_query(query_df)
    all_results.append(result)

    # Run structural audit
    audit = structural_audit(result['generated_note'], result['formatted_records'])

    eval_row = {
        'query_id': result['query_id'],
        'query_text': result['query_text'][:60],
        'n_records': result['n_records'],
        'latency_seconds': round(result['latency_seconds'], 1),
        'note_length': len(result['generated_note']),
        'structural_pass': audit['pass'],
        'structural_issues': audit['issue_count'],
        **audit['scores'],
    }
    all_evals.append(eval_row)

    status = "PASS" if audit['pass'] else "FAIL"
    print(f"  Records: {result['n_records']} | "
          f"Time: {result['latency_seconds']:.1f}s | "
          f"Structural: {status} | "
          f"Note: {len(result['generated_note']):,} chars")

print(f"\n{'='*80}")
print(f"Complete: {len(all_results)} queries processed.")

Total queries to process: 6

[1/6] Q1_002: PW2XZT68KVW9, PW2XZT69KVW8, PW 2XZT68KVW9...


  Records: 3 | Time: 49.1s | Structural: PASS | Note: 4,186 chars
[2/6] Q2_002: Tovarystvo z obmezhenoiu vidpovidalnistiu "Zelinskyi Hrupp",...


  Records: 3 | Time: 43.7s | Structural: PASS | Note: 3,993 chars
[3/6] Q3_002: North Korean state-sponsored hacking group that stole crypto...


  Records: 10 | Time: 44.1s | Structural: PASS | Note: 4,928 chars
[4/6] Q4_002: What entities are connected to the Lazarus Group through fro...


  Records: 10 | Time: 56.0s | Structural: PASS | Note: 6,590 chars
[5/6] Q5_002: Per Atle Tufte, per atle tufte, Per Atle...


  Records: 3 | Time: 37.9s | Structural: PASS | Note: 3,808 chars
[6/6] Q6_002: BR CEIS company Brazil, BR CEIS entity Brazil, BR CEIS compa...


  Records: 3 | Time: 41.4s | Structural: PASS | Note: 3,637 chars

Complete: 6 queries processed.


---
## 9 · View Generated Notes

Browse the generated triage notes. Change the index to view different queries.

In [10]:
# ---- View a specific triage note ----
VIEW_INDEX = 0  # Change this to view different queries

if all_results:
    r = all_results[VIEW_INDEX]
    print(f"{'='*80}")
    print(f"QUERY: {r['query_text']}")
    print(f"  Type: {r['query_type']} | Records: {r['n_records']} | Time: {r['latency_seconds']:.1f}s")
    print(f"{'='*80}")
    print(r['generated_note'])
    print(f"{'='*80}")

QUERY: PW2XZT68KVW9, PW2XZT69KVW8, PW 2XZT68KVW9
  Type: Type 1 | Records: 3 | Time: 49.1s
### CRITERIA EXTRACTION

First, break down the query into its individual criteria:
- **Criterion 1:** The entity’s identifier (ID) must match one of the search terms: *PW2XZT68KVW9*, *PW2XZT69KVW8*, or *PW 2XZT68KVW9*.
- **Criterion 2:** The query type is “Type 1”, implying that the entity should be listed in a sanctions‑related registry (e.g., a sanctions list or public‑officials database).  
- **Criterion 3:** The entity must be a person or company that is currently active (i.e., has a “Last Seen” date within the last year).

---

### PER-RECORD ASSESSMENT

**Record 1: Pavol Mertus | Person | Slovakia**

Criteria met:
- **Criterion 1:** NOT DETERMINABLE — No identifier field is present in the record to compare with the search terms.  
- **Criterion 2:** NOT DETERMINABLE — The record indicates the entity is listed in *sk_public_officials*, but the query type “Type 1” is not explicitly defined in

---
## 10 · RAGAS Evaluation (Optional)

RAGAS requires a paid cloud API (OpenAI or Anthropic) for evaluation.
Currently skipped. Structural audit provides the primary evaluation.

To enable later: set `EVAL_LLM_MODEL = 'gpt-4o'` and provide `OPENAI_API_KEY`.

In [11]:
# RAGAS evaluation — skipped (requires paid API)
if EVAL_LLM_MODEL is not None:
    print("RAGAS evaluation would run here with:", EVAL_LLM_MODEL)
    print("Not implemented in this offline-only notebook.")
else:
    print("RAGAS skipped (EVAL_LLM_MODEL = None).")
    print("Structural audit results available in the summary below.")

RAGAS skipped (EVAL_LLM_MODEL = None).
Structural audit results available in the summary below.


---
## 11 · Evaluation Summary

In [12]:
# ======================================================================
# Summary
# ======================================================================

df_eval = pd.DataFrame(all_evals)

print("=" * 80)
print("  RAG TRIAGE EVALUATION SUMMARY")
print("=" * 80)
print()

print("AGGREGATE METRICS:")
print(f"  Queries processed      : {len(df_eval)}")
print(f"  Structural pass rate   : {df_eval['structural_pass'].mean():.1%}")
print(f"  Mean structural score  : {df_eval['structural_score'].mean():.4f}")
if 'faithfulness' in df_eval.columns:
    print(f"  Mean Faithfulness      : {df_eval['faithfulness'].mean():.4f}")
else:
    print(f"  Faithfulness           : N/A (RAGAS not configured)")
print(f"  Mean latency           : {df_eval['latency_seconds'].mean():.1f}s")
print(f"  Mean note length       : {df_eval['note_length'].mean():,.0f} chars")
print()

print("PER-QUERY BREAKDOWN:")
print("-" * 80)
show_cols = ['query_id', 'n_records', 'structural_pass', 'structural_score',
             'latency_seconds', 'note_length']
if 'faithfulness' in df_eval.columns:
    show_cols.insert(4, 'faithfulness')
available = [c for c in show_cols if c in df_eval.columns]
print(df_eval[available].to_string(index=False))

  RAG TRIAGE EVALUATION SUMMARY

AGGREGATE METRICS:
  Queries processed      : 6
  Structural pass rate   : 100.0%
  Mean structural score  : 1.0000
  Faithfulness           : N/A (RAGAS not configured)
  Mean latency           : 45.4s
  Mean note length       : 4,524 chars

PER-QUERY BREAKDOWN:
--------------------------------------------------------------------------------
query_id  n_records  structural_pass  structural_score  latency_seconds  note_length
  Q1_002          3             True               1.0             49.1         4186
  Q2_002          3             True               1.0             43.7         3993
  Q3_002         10             True               1.0             44.1         4928
  Q4_002         10             True               1.0             56.0         6590
  Q5_002          3             True               1.0             37.9         3808
  Q6_002          3             True               1.0             41.4         3637


---
## 12 · Export Results

In [13]:
# ======================================================================
# Export
# ======================================================================
output_dir = Path('rag_output')
output_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Rebuild df_eval in case summary cell was skipped
df_eval = pd.DataFrame(all_evals)

# 1. Evaluation scores
eval_path = output_dir / f'rag_eval_{timestamp}.csv'
df_eval.to_csv(eval_path, index=False)
# ======================================================================
# Export
# ======================================================================

output_dir = Path('rag_output')
output_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# 1. Evaluation scores
eval_path = output_dir / f'rag_eval_{timestamp}.csv'
df_eval.to_csv(eval_path, index=False)
print(f"Evaluation scores : {eval_path}")

# 2. All generated notes
notes_path = output_dir / f'rag_notes_{timestamp}.json'
notes_export = []
for result, ev in zip(all_results, all_evals):
    notes_export.append({
        'query_id': result['query_id'],
        'query_text': result['query_text'],
        'query_type': result['query_type'],
        'n_records': result['n_records'],
        'generated_note': result['generated_note'],
        'latency_seconds': result['latency_seconds'],
        'evaluation': ev,
    })

with open(notes_path, 'w', encoding='utf-8') as f:
    json.dump(notes_export, f, indent=2, ensure_ascii=False, default=str)
print(f"Triage notes      : {notes_path}")

# 3. Individual notes as text files (easy to read)
notes_dir = output_dir / 'notes'
notes_dir.mkdir(exist_ok=True)
for result in all_results:
    note_file = notes_dir / f"{result['query_id']}.txt"
    with open(note_file, 'w', encoding='utf-8') as f:
        f.write(f"QUERY: {result['query_text']}\n")
        f.write(f"TYPE: {result['query_type']}\n")
        f.write(f"RECORDS: {result['n_records']}\n")
        f.write(f"{'='*60}\n\n")
        f.write(result['generated_note'])
print(f"Individual notes  : {notes_dir}/ ({len(all_results)} files)")

print(f"\nAll outputs in: {output_dir}/")

Evaluation scores : rag_output\rag_eval_20260410_091700.csv
Triage notes      : rag_output\rag_notes_20260410_091700.json
Individual notes  : rag_output\notes/ (6 files)

All outputs in: rag_output/


---
## 13 · Next Steps

1. **Review generated notes** — Open the `.txt` files in `rag_output/notes/`
   and verify the triage notes make sense against the pooling data.

2. **Enable RAGAS** — Set `EVAL_LLM_MODEL = 'gpt-4o'` or `'claude-sonnet-4-20250514'`
   with an API key to get Faithfulness scores.

3. **Tune the prompt** — If structural audit failures are common, adjust the
   system prompt rules or output format.

4. **Compare LLMs** — Run the same queries through different models
   (llama3.1 vs mistral vs GPT-4o) and compare structural scores.

5. **Add ground truth** — Have an analyst label classifications as correct/incorrect
   to measure classification accuracy (Layer 3 evaluation).

In [14]:
# ======================================================================
# Build per-record RAG columns and merge back into original data
# ======================================================================

from datetime import datetime
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Rebuild df_eval in case summary cell was skipped
df_eval = pd.DataFrame(all_evals)

# --- Step 1: Build a lookup of per-record notes ---
record_notes = {}   # (query_id, doc_id) -> note text
record_class = {}   # (query_id, doc_id) -> extracted classification

for result in all_results:
    qid = result['query_id']
    query_df = df[df['query_id'] == qid].head(MAX_RECORDS_PER_QUERY)

    # Split the combined note back into per-record sections
    sections = re.split(r'### Result \d+:', result['generated_note'])
    sections = [s.strip() for s in sections[1:] if s.strip()]

    # Also try alternate split patterns the LLM might use
    if not sections:
        sections = re.split(r'\*\*Result \d+', result['generated_note'])
        sections = [s.strip() for s in sections[1:] if s.strip()]

    if not sections:
        sections = re.split(r'Result \d+:', result['generated_note'])
        sections = [s.strip() for s in sections[1:] if s.strip()]

    for idx, (_, row) in enumerate(query_df.iterrows()):
        did = str(row.get('doc_id', ''))
        key = (qid, did)

        if idx < len(sections):
            note = sections[idx]
            note_lines = note.split('\n', 1)
            if len(note_lines) > 1:
                record_notes[key] = note_lines[1].strip()
            else:
                record_notes[key] = note.strip()

            cls_match = re.search(
                r'(TRUE POSITIVE|AMBIGUOUS|LIKELY FALSE POSITIVE)',
                note, re.IGNORECASE
            )
            record_class[key] = cls_match.group(0).upper() if cls_match else ''
        else:
            # If LLM didn't produce per-record sections, store full note for first record
            if idx == 0 and not sections:
                record_notes[key] = result['generated_note']
                cls_match = re.search(
                    r'(TRUE POSITIVE|AMBIGUOUS|LIKELY FALSE POSITIVE)',
                    result['generated_note'], re.IGNORECASE
                )
                record_class[key] = cls_match.group(0).upper() if cls_match else ''
            else:
                record_notes[key] = ''
                record_class[key] = ''

# --- Step 2: Build evaluation lookup (query-level) ---
eval_lookup = {}
for ev in all_evals:
    eval_lookup[ev['query_id']] = ev

# --- Step 3: Add new columns to original DataFrame ---
df_enriched = df.copy()

df_enriched['rag_note'] = df_enriched.apply(
    lambda r: record_notes.get((str(r['query_id']), str(r.get('doc_id', ''))), ''),
    axis=1
)

df_enriched['rag_classification'] = df_enriched.apply(
    lambda r: record_class.get((str(r['query_id']), str(r.get('doc_id', ''))), ''),
    axis=1
)

df_enriched['rag_model'] = LLM_MODEL

df_enriched['rag_latency_sec'] = df_enriched['query_id'].map(
    {r['query_id']: round(r['latency_seconds'], 2) for r in all_results}
)

df_enriched['rag_structural_pass'] = df_enriched['query_id'].map(
    {ev['query_id']: ev.get('structural_pass', '') for ev in all_evals}
)

df_enriched['rag_structural_score'] = df_enriched['query_id'].map(
    {ev['query_id']: ev.get('structural_score', '') for ev in all_evals}
)

df_enriched['rag_timestamp'] = timestamp

print(f"Enriched DataFrame: {len(df_enriched):,} rows x {len(df_enriched.columns)} columns")
print(f"New columns added: rag_note, rag_classification, rag_model, "
      f"rag_latency_sec, rag_structural_pass, rag_structural_score, rag_timestamp")
print(f"\nRAG note coverage: {(df_enriched['rag_note'] != '').sum():,} / {len(df_enriched):,} rows have notes")
print(f"Classifications found: {(df_enriched['rag_classification'] != '').sum():,} rows")

# Preview
has_notes = df_enriched[df_enriched['rag_note'] != '']
if len(has_notes) > 0:
    print("\n--- Sample enriched row ---")
    sample_idx = has_notes.index[0]
    for col in ['query_id', 'doc_id', 'caption', 'rag_classification', 'rag_structural_pass', 'rag_note']:
        val = df_enriched.loc[sample_idx, col]
        val_str = str(val)[:100]
        print(f"  {col:25s}: {val_str}")
else:
    print("\nWARNING: No RAG notes were matched to records.")
    print("  This may mean the LLM output format didn't include '### Result N:' headers.")
    print("  Check the raw notes in all_results[0]['generated_note']")


# ======================================================================
# Export enriched Excel file
# ======================================================================

output_dir = Path('rag_output')
output_dir.mkdir(exist_ok=True)

input_stem = rag_input_path.stem
out_filename = f'{input_stem}_RAG_enriched.xlsx'
out_path = output_dir / out_filename

# Column ordering: original columns first, then RAG columns at the end
original_cols = [c for c in df.columns if c in df_enriched.columns]
rag_cols = ['rag_classification', 'rag_note', 'rag_model',
            'rag_latency_sec', 'rag_structural_pass', 'rag_structural_score',
            'rag_timestamp']
all_cols = original_cols + [c for c in rag_cols if c in df_enriched.columns]

df_export = df_enriched[all_cols]

with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
    df_export.to_excel(writer, index=False, sheet_name='rag_enriched')

    ws = writer.sheets['rag_enriched']
    col_widths = {
        'query_id': 12, 'query_text': 50, 'query_type': 14,
        'query_notes': 30, 'doc_id': 28, 'caption': 30,
        'rag_classification': 22, 'rag_note': 80,
        'rag_model': 22, 'rag_latency_sec': 14,
        'rag_structural_pass': 18, 'rag_structural_score': 18,
        'rag_timestamp': 20, 'embedding_text': 60, 'text_blob': 60,
        'meta_country': 12, 'schema': 12, 'meta_programId': 30,
    }
    for col_idx, col_name in enumerate(all_cols, 1):
        width = col_widths.get(col_name, 15)
        letter = ws.cell(row=1, column=col_idx).column_letter
        ws.column_dimensions[letter].width = width
    ws.freeze_panes = 'A2'

    # Summary sheet
    summary_data = []
    for ev in all_evals:
        summary_data.append({
            'query_id': ev['query_id'],
            'query_text': ev.get('query_text', '')[:60],
            'n_records': ev.get('n_records', 0),
            'structural_pass': ev.get('structural_pass', ''),
            'structural_score': ev.get('structural_score', ''),
            'latency_seconds': ev.get('latency_seconds', ''),
            'note_length': ev.get('note_length', ''),
        })
    df_summary = pd.DataFrame(summary_data)
    df_summary.to_excel(writer, index=False, sheet_name='rag_eval_summary')
    ws2 = writer.sheets['rag_eval_summary']
    ws2.freeze_panes = 'A2'
    for col_idx in range(1, len(df_summary.columns) + 1):
        ws2.column_dimensions[ws2.cell(row=1, column=col_idx).column_letter].width = 18

print(f"{'='*60}")
print(f"  ENRICHED EXCEL EXPORTED")
print(f"{'='*60}")
print(f"  File: {out_path}")
print(f"  Rows: {len(df_export):,}")
print(f"  Columns: {len(all_cols)} ({len(original_cols)} original + {len(rag_cols)} RAG)")
print(f"  Sheets: 'rag_enriched' (full data) + 'rag_eval_summary' (per-query scores)")
print(f"{'='*60}")
print()
print("New RAG columns appended to original data:")
for col in rag_cols:
    if col in df_export.columns:
        non_empty = (df_export[col].astype(str) != '').sum()
        print(f"  {col:25s}: {non_empty:,} non-empty values")

Enriched DataFrame: 32 rows x 32 columns
New columns added: rag_note, rag_classification, rag_model, rag_latency_sec, rag_structural_pass, rag_structural_score, rag_timestamp

RAG note coverage: 6 / 32 rows have notes
Classifications found: 0 rows

--- Sample enriched row ---
  query_id                 : Q1_002
  doc_id                   : sk-po-354d893265060ef749116a7722d8880e06000be5
  caption                  : Pavol Mertus
  rag_classification       : 
  rag_structural_pass      : True
  rag_note                 : ### CRITERIA EXTRACTION

First, break down the query into its individual criteria:
- **Criterion 1:*
  ENRICHED EXCEL EXPORTED
  File: rag_output\08_(RAG)_top_10_fused_with_full_data_20260410_091119_RAG_enriched.xlsx
  Rows: 32
  Columns: 32 (25 original + 7 RAG)
  Sheets: 'rag_enriched' (full data) + 'rag_eval_summary' (per-query scores)

New RAG columns appended to original data:
  rag_classification       : 0 non-empty values
  rag_note                 : 6 non-empty va